In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA gold;


#Transformation Logic

In [0]:
query = """
SELECT
  ROW_NUMBER() OVER (ORDER BY c.customer_id) AS customer_key,
  c.customer_key,
  c.customer_id,
  c.first_name,
  c.last_name,
  l.country,
  c.marital_status,
  COALESCE(NULLIF(c.gender, 'Unknown'), e.gender) AS gender,
  e.birth_date,
  c.cst_create_date
FROM silver.crm_customers c
LEFT JOIN silver.erp_customers e
  ON c.customer_key = e.customer_id
LEFT JOIN silver.erp_customers_location l
  ON c.customer_key = l.customer_id
"""
df = spark.sql(query)
df.display()


#Writing Gold Table

In [0]:
df.write.mode("overwrite").saveAsTable("gold.dim_customers")


In [0]:
%sql
select * from workspace.gold.dim_customers

#Sanity Checks

In [0]:
%sql
SELECT customer_key, COUNT(*)
FROM workspace.gold.dim_customers
GROUP BY customer_key
HAVING COUNT(*) > 1;


In [0]:
%sql
COMMENT ON TABLE workspace.gold.dim_customers IS
'Customer dimension table containing one record per customer, enriched from CRM and ERP systems. Used for customer-level analysis.';
